In [1]:
import numpy as np
import torch
import torch.nn as nn

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

In [2]:
data = np.load("pems04.npz")

traffic = data["data"][:,:,2]

print(traffic.shape)

(16992, 307)


In [3]:
scaler = MinMaxScaler()

traffic_scaled = scaler.fit_transform(
    traffic
)

print(
    traffic_scaled.shape
)

(16992, 307)


In [4]:
X = []
y = []

sequence_length = 12

for i in range(
    len(traffic_scaled)
    -
    sequence_length
):

    X.append(
        traffic_scaled[
            i:i+sequence_length
        ]
    )

    y.append(
        traffic_scaled[
            i+sequence_length
        ]
    )

X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)

(16980, 12, 307)
(16980, 307)


In [5]:
split = int(
    len(X) * 0.8
)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

print(X_train.shape)
print(X_test.shape)

(13584, 12, 307)
(3396, 12, 307)


In [6]:
X_train = torch.tensor(
    X_train,
    dtype=torch.float32
)

X_test = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_train = torch.tensor(
    y_train,
    dtype=torch.float32
)

y_test = torch.tensor(
    y_test,
    dtype=torch.float32
)

In [7]:
train_dataset = TensorDataset(
    X_train,
    y_train
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

In [8]:
import torch.nn as nn

class TemporalConv(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels
    ):

        super().__init__()

        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=(3,1),
            padding=(1,0)
        )

    def forward(self,x):

        return torch.relu(
            self.conv(x)
        )

In [9]:
class AttentionGraphConv(nn.Module):

    def __init__(
        self,
        num_nodes,
        channels
    ):

        super().__init__()

        self.node_embeddings = nn.Parameter(
            torch.randn(
                num_nodes,
                16
            )
        )

        self.attention = nn.Linear(
            16,
            16
        )

        self.weight = nn.Linear(
            channels,
            channels
        )

    def forward(self,x):

        E = self.attention(
            self.node_embeddings
        )

        adj = torch.softmax(
            torch.relu(
                E @ E.T
            ),
            dim=1
        )

        x = torch.einsum(
            "ij,bctj->bcti",
            adj,
            x
        )

        x = x.permute(
            0,
            2,
            3,
            1
        )

        x = self.weight(x)

        x = x.permute(
            0,
            3,
            1,
            2
        )

        return torch.relu(x)

In [10]:
class AGCSTGCN(nn.Module):

    def __init__(self):

        super().__init__()

        self.temp1 = TemporalConv(
            1,
            32
        )

        self.graph = AttentionGraphConv(
            num_nodes=307,
            channels=32
        )

        self.temp2 = TemporalConv(
            32,
            32
        )

        self.fc = nn.Linear(
            32,
            1
        )

    def forward(self,x):

        x = x.unsqueeze(1)

        x = self.temp1(x)

        x = self.graph(x)

        x = self.temp2(x)

        x = x.mean(dim=2)

        x = x.permute(
            0,
            2,
            1
        )

        x = self.fc(x)

        return x.squeeze(-1)

In [11]:
model = AGCSTGCN()

X_batch, y_batch = next(
    iter(train_loader)
)

pred = model(X_batch)

print(pred.shape)
print(y_batch.shape)

torch.Size([64, 307])
torch.Size([64, 307])


In [12]:
model = AGCSTGCN()

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

import time

train_start = time.time()

epochs = 30

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        pred = model(X_batch)

        loss = criterion(
            pred,
            y_batch
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{epochs} "
        f"Loss: {total_loss/len(train_loader):.6f}"
    )

Epoch 1/30 Loss: 0.083583
Epoch 2/30 Loss: 0.003651
Epoch 3/30 Loss: 0.001805
Epoch 4/30 Loss: 0.001559
Epoch 5/30 Loss: 0.001437
Epoch 6/30 Loss: 0.001354
Epoch 7/30 Loss: 0.001281
Epoch 8/30 Loss: 0.001217
Epoch 9/30 Loss: 0.001166
Epoch 10/30 Loss: 0.001125
Epoch 11/30 Loss: 0.001087
Epoch 12/30 Loss: 0.001049
Epoch 13/30 Loss: 0.001025
Epoch 14/30 Loss: 0.001020
Epoch 15/30 Loss: 0.001011
Epoch 16/30 Loss: 0.000994
Epoch 17/30 Loss: 0.000990
Epoch 18/30 Loss: 0.000988
Epoch 19/30 Loss: 0.000989
Epoch 20/30 Loss: 0.000977
Epoch 21/30 Loss: 0.000983
Epoch 22/30 Loss: 0.000971
Epoch 23/30 Loss: 0.000971
Epoch 24/30 Loss: 0.000967
Epoch 25/30 Loss: 0.000973
Epoch 26/30 Loss: 0.000977
Epoch 27/30 Loss: 0.000963
Epoch 28/30 Loss: 0.000962
Epoch 29/30 Loss: 0.000963
Epoch 30/30 Loss: 0.000963


In [13]:
train_time = time.time() - train_start

print("Time Taken:", train_time)

Time Taken: 1510.168030500412


In [14]:
torch.save(
    model.state_dict(),
    "AGC-STGCN-PEMS04.pth"
)

In [15]:
model.eval()

infer_start = time.time()

with torch.no_grad():

    predictions = model(
        X_test
    )

predictions = predictions.numpy()

infer_time = time.time() - infer_start
print("Infer Time:", infer_time)

true_values = y_test.numpy()

mae = mean_absolute_error(
    true_values,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        true_values,
        predictions
    )
)

print("MAE:", mae)
print("RMSE:", rmse)

Infer Time: 29.365830659866333
MAE: 0.018758511170744896
RMSE: 0.03279593012435959


In [16]:
from sklearn.metrics import r2_score

mape = np.mean(
    np.abs((true_values - predictions) /
           np.maximum(np.abs(true_values), 1e-6))
) * 100

r2 = r2_score(
    true_values.flatten(),
    predictions.flatten()
)
print("MAPE:", mape)
print("R2:", r2)

MAPE: 1180.6046
R2: 0.9488536715507507


In [17]:
params = sum(
    p.numel()
    for p in model.parameters()
)

print("Parameters:", params)

Parameters: 9505
